In [1]:
!pip install -q \
langgraph \
langchain \
langchain-openai \
langchain-community \
langchain-chroma \
langchain-text-splitters \
chromadb \
pypdf \
python-dotenv

환경변수

In [3]:
import os
from dotenv import load_dotenv

load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

라이브러리

In [4]:
from typing import TypedDict
from langgraph.graph import StateGraph
from langgraph.graph import END
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

LLM

In [5]:
llm = ChatOpenAI(
    model="gpt-5.4-nano",
    temperature=0
)

embedding = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

Vector DB 연결

In [6]:
law_db = Chroma(
    persist_directory="vector_db/laws",
    embedding_function=embedding
)

precedent_db = Chroma(
    persist_directory="vector_db/precedents",
    embedding_function=embedding
)

State

In [7]:
class GraphState(TypedDict):
    
    question: str

    law_docs: list
    law_analysis: str

    precedent_docs: list
    precedent_analysis: str

    final_answer: str

법령 검색 노드

In [8]:
def retrieve_law_node(state):

    question = state["question"]

    docs = law_db.similarity_search(
        question,
        k=5
    )

    return {
        "law_docs": docs
    }

법령 분석 노드

In [9]:
def analyze_law_node(state):

    question = state["question"]

    # 조문 단위 메타데이터로 출처 라벨 구성
    law_context_parts = []
    for doc in state["law_docs"]:
        m = doc.metadata
        law_name   = m.get("law_name", "")
        article_no = m.get("article_no", "")
        article_title = m.get("article_title", "")
        chapter    = m.get("chapter", "")
        source_pdf = m.get("source_pdf", "")
        page       = m.get("page", "")

        # "근로기준법 / 제3장 제43조 (임금 지급) / 1p" 형태
        parts = [p for p in [law_name, chapter, f"{article_no} ({article_title})" if article_title else article_no] if p]
        label = " / ".join(parts)
        if page:
            label += f" / {page}p"
        if source_pdf:
            label += f" [{source_pdf}]"

        law_context_parts.append(f"[출처: {label}]\n{doc.page_content}")

    law_context = "\n\n".join(law_context_parts)

    prompt = f"""
당신은 대한민국 노동법 전문 AI입니다.

사용자 사례:
{question}

관련 법령 (각 조문 앞에 출처 표시):
{law_context}

규칙

1. 관련 법률만 설명
2. 위법이라고 단정 금지
3. 반드시 법률명, 장(章), 조(條), 항(項)을 명시 (예: 근로기준법 제3장 제43조 제1항)
4. 조문이 확인되지 않으면 "조문 불명확"으로 표시
5. 출처 PDF 파일명도 함께 표시

출력 형식 (반드시 준수)

관련 법률:
- 법률명 제X장 제X조 제X항 (출처: [파일명] Xp): 조항 내용 요약

설명:
- 해당 조항이 사용자 사례에 적용되는 이유
"""

    result = llm.invoke(prompt).content

    return {
        "law_analysis": result
    }


판례 검색 노드

In [ ]:
def retrieve_precedent_node(state):

    # 핵심 쟁점만 추출
    keyword_prompt = f"""
다음 사용자 사례와 법률 분석에서
판례 검색에 사용할 핵심 법률 쟁점 키워드를 
10단어 이내로 추출하세요.

사용자 사례: {state['question']}
법률 분석 요약: {state['law_analysis'][:300]}

출력: 키워드만, 문장 금지
"""
    search_query = llm.invoke(keyword_prompt).content.strip()

    docs = precedent_db.similarity_search(search_query, k=5)

    return {"precedent_docs": docs}

판례 분석 노드

In [11]:
def analyze_precedent_node(state):

    # 판례 메타데이터: source_file (예: "2011다23149.md")
    precedent_context_parts = []
    for doc in state["precedent_docs"]:
        m = doc.metadata
        source_file = m.get("source_file", "알 수 없는 판례")
        # 파일명에서 사건번호 추출 (확장자 제거)
        case_no = source_file.replace(".md", "").replace(".json", "")
        label = f"[판례: {case_no}]"
        precedent_context_parts.append(f"{label}\n{doc.page_content}")

    precedent_context = "\n\n".join(precedent_context_parts)

    prompt = f"""
사용자 사례:
{state['question']}

검색된 판례 (각 판례 앞에 출처 표시):
{precedent_context}

규칙

1. 반드시 판례명(사건번호) 전체 표기 (예: 대법원 2019. 4. 18. 선고 2016다2451 판결)
2. 판결 요지 1~2문장으로 핵심 요약
3. 사용자 사례와의 유사점을 구체적으로 설명
4. 법률 자문처럼 단정 금지
5. 출처 파일명(사건번호) 반드시 표시

출력 형식 (반드시 준수)

판례명: [전체 사건번호 포함]
출처: [파일명]
판결 요지: [핵심 내용 1~2문장]
사용자 사례 관련성: [구체적 유사점]
"""

    result = llm.invoke(prompt).content

    return {
        "precedent_analysis": result
    }


최종 답변 노드

In [12]:
def generate_answer_node(state):

    prompt = f"""
사용자 사례:
{state["question"]}

법률 분석:
{state["law_analysis"]}

판례 분석:
{state["precedent_analysis"]}

당신은 노동법 분석 시스템이다.

반드시 아래 형식만 출력하라.

# 관련 법률

- 법률명 및 조항: 예) 근로기준법 제3장 제43조 제1항
- 조항 내용 요약
- 적용 가능성

# 관련 판례

- 판례명: 예) 대법원 2019. 4. 18. 선고 2016다2451 판결
- 출처: 판례 파일명 / 분류
- 판결 요지
- 사용자 사례와의 관련성

# 결론

- 현재 정보 기준 분석 결과
- 주요 근거 조항 재정리 (법률명 + 조번호)

규칙

1. 추가 질문 금지
2. 추가 정보 요청 금지
3. 상담 권유 금지
4. "원하시면", "추가로 알려주시면" 같은 문장 금지
5. 위 형식 외 내용 출력 금지
6. 최종 결과만 작성
7. 조항 번호가 불명확한 경우 "조항 확인 필요"로 표시
"""

    answer = llm.invoke(prompt).content

    return {
        "final_answer": answer
    }


Graph 생성

In [13]:
builder = StateGraph(GraphState)

builder.add_node(
    "retrieve_law",
    retrieve_law_node
)

builder.add_node(
    "analyze_law",
    analyze_law_node
)

builder.add_node(
    "retrieve_precedent",
    retrieve_precedent_node
)

builder.add_node(
    "analyze_precedent",
    analyze_precedent_node
)

builder.add_node(
    "generate_answer",
    generate_answer_node
)

Edge 연결

In [14]:
builder.set_entry_point(
    "retrieve_law"
)

builder.add_edge(
    "retrieve_law",
    "analyze_law"
)

builder.add_edge(
    "analyze_law",
    "retrieve_precedent"
)

builder.add_edge(
    "retrieve_precedent",
    "analyze_precedent"
)

builder.add_edge(
    "analyze_precedent",
    "generate_answer"
)

builder.add_edge(
    "generate_answer",
    END
)

graph = builder.compile()

법령 PDF 벡터DB 생성 (조문 단위 파싱 — law_parser.py 사용)

In [15]:
import json
from pathlib import Path
from langchain_core.documents import Document
from langchain_chroma import Chroma

law_root = Path("data/process/law")

# ── 경로 및 파일 존재 여부 확인 ──────────────────────────────
if not law_root.exists():
    raise FileNotFoundError(
        f"폴더가 없습니다: {law_root.resolve()}\n"
        "law_parser.py의 process_all_pdfs()를 먼저 실행해 주세요."
    )

json_files = sorted(law_root.rglob("*.json"))
if not json_files:
    raise FileNotFoundError(
        f"{law_root} 안에 JSON 파일이 없습니다.\n"
        "law_parser.py의 process_all_pdfs()를 먼저 실행해 주세요."
    )

print(f"JSON 파일 {len(json_files)}개 발견:")
for jf in json_files:
    print(f"  {jf}")

# ── JSON → Document 로드 ─────────────────────────────────────
all_law_docs = []
for json_file in json_files:
    with open(json_file, "r", encoding="utf-8") as f:
        items = json.load(f)
    for item in items:
        all_law_docs.append(
            Document(
                page_content=item["page_content"],
                metadata=item["metadata"]
            )
        )

if not all_law_docs:
    raise ValueError("로드된 Document가 0개입니다. JSON 파일 내용을 확인해 주세요.")

print(f"\n총 {len(all_law_docs)}개 조문 Document 로드 완료")
print("\n[메타데이터 예시]")
for k, v in all_law_docs[0].metadata.items():
    print(f"  {k}: {v}")

# ── Vector DB 저장 ───────────────────────────────────────────
law_db = Chroma.from_documents(
    documents=all_law_docs,
    embedding=embedding,
    persist_directory="vector_db/laws"
)

print("\n법령 DB 생성 완료")


JSON 파일 24개 발견:
  data\process\law\근로기준법\근로기준법 시행규칙(고용노동부령)(제00436호)(20250223).json
  data\process\law\근로기준법\근로기준법 시행령(대통령령)(제35436호)(20251023).json
  data\process\law\근로기준법\근로기준법(법률)(제20520호)(20251023) (1).json
  data\process\law\근로자퇴직급여 보장법\근로자퇴직급여 보장법 시행규칙(고용노동부령)(제00359호)(20220712).json
  data\process\law\근로자퇴직급여 보장법\근로자퇴직급여 보장법 시행령(대통령령)(제36220호)(20260324).json
  data\process\law\근로자퇴직급여 보장법\근로자퇴직급여 보장법(법률)(제21135호)(20251111).json
  data\process\law\기간제 및 단기근로자 보호등에 관한 법률\기간제 및 단시간근로자 보호 등에 관한 법률 시행규칙(노동부령)(제00277호)(20070701).json
  data\process\law\기간제 및 단기근로자 보호등에 관한 법률\기간제 및 단시간근로자 보호 등에 관한 법률 시행령(대통령령)(제31611호)(20210408).json
  data\process\law\기간제 및 단기근로자 보호등에 관한 법률\기간제 및 단시간근로자 보호 등에 관한 법률(법률)(제18177호)(20210518).json
  data\process\law\남녀고용평등과 일가정 양립 지원에 관한 법률\남녀고용평등과 일ㆍ가정 양립 지원에 관한 법률 시행규칙(고용노동부령)(제00453호)(20251001).json
  data\process\law\남녀고용평등과 일가정 양립 지원에 관한 법률\남녀고용평등과 일ㆍ가정 양립 지원에 관한 법률 시행령(대통령령)(제35806호)(20251001).json
  data\process\law\남녀고용평등과 일가정 양립 지원에 관한 법률\남녀고용평등과

판례 벡터DB 생성

In [16]:
import json
from pathlib import Path
from langchain_core.documents import Document
from langchain_chroma import Chroma

precedent_root = Path("data/process/판례")
all_docs = []

# JSON 파일로 저장된 판례 로드
for json_file in sorted(precedent_root.rglob("*.json")):

    print(f"로딩 중: {json_file.name}")

    with open(json_file, "r", encoding="utf-8") as f:
        items = json.load(f)

    # 파일 하나에 판례 여러 건이 리스트로 담겨 있을 수 있음
    if isinstance(items, dict):
        items = [items]

    for item in items:
        doc = Document(
            page_content=item["page_content"],
            metadata=item.get("metadata", {})
        )
        all_docs.append(doc)

print(f"\n총 {len(all_docs)}개 판례 Document 로드 완료")
if all_docs:
    print("[메타데이터 예시]", all_docs[0].metadata)

# 판례는 1건이 하나의 Document — 청크 분할 없음
precedent_db = Chroma.from_documents(
    documents=all_docs,
    embedding=embedding,
    persist_directory="vector_db/precedents"
)

print("판례 DB 생성 완료")

로딩 중: 2011다23149.json
로딩 중: 2006다64245.json
로딩 중: 2014다44673.json
로딩 중: 2018도6486.json
로딩 중: 2001도204.json
로딩 중: 2017도4343.json
로딩 중: 2022도2188.json
로딩 중: 98도1260.json
로딩 중: 2010다91046.json
로딩 중: 2012다89399.json
로딩 중: 2015다73067.json
로딩 중: 89다카2292.json
로딩 중: 94다19501.json
로딩 중: 2022두64518.json
로딩 중: 90누2772.json
로딩 중: 97다5015.json
로딩 중: 2012다106423.json
로딩 중: 2015다73067.json
로딩 중: 2020도15393.json
로딩 중: 2008다57852.json
로딩 중: 2008다6052.json
로딩 중: 96다24699.json
로딩 중: 2007다90760.json
로딩 중: 2016다255910.json
로딩 중: 93다26168.json
로딩 중: 97누18189.json
로딩 중: 2002다11458.json
로딩 중: 90다11554.json
로딩 중: 91다36666.json
로딩 중: 2020도16541.json
로딩 중: 2004다29736.json
로딩 중: 2006도777.json
로딩 중: 2009다51417.json
로딩 중: 2002다62432.json
로딩 중: 2019두55859.json
로딩 중: 2006두4912.json
로딩 중: 2017두45933.json
로딩 중: 2019두62604.json
로딩 중: 2020다270503.json
로딩 중: 2007두22498.json
로딩 중: 2017두74702.json
로딩 중: 2012두4852.json
로딩 중: 2015두51651.json
로딩 중: 2017두76005.json
로딩 중: 2018두47264.json
로딩 중: 2019두38571.json

총 46개 판례 Document

In [18]:
result = graph.invoke(
    {
        "question":
        "회사와 근로계약서를 체결하여 선수단에서 활동하는 장애인 아마추어 운동 선수가 「근로기준법」상 근로자에 해당하는지"
    }
)

print(result["final_answer"])

# 관련 법률

- 법률명 및 조항: 파견근로자 보호 등에 관한 법률(파견법) 제3장 제34조 제1항  
- 조항 내용 요약: 파견 중인 근로자에 대해 파견사업주와 사용사업주를 「근로기준법」상 사용자로 보아, 그에 맞게 「근로기준법」을 적용하되 사용자 판단을 주체별로 구분함.  
- 적용 가능성: “선수단 활동”이 형식상 도급/위탁처럼 보여도 실질이 근로자 파견 또는 사용종속관계로 평가될 여지가 있으면, 「근로기준법」상 사용자 판단과 적용범위 판단에 참고됨(선수=근로자 자동 단정은 아님).

- 법률명 및 조항: 근로기준법 제9장 제93조 제1항  
- 조항 내용 요약: 상시 10명 이상 근로자를 사용하는 사용자는 취업규칙을 작성·신고해야 하며, 근로시간/휴일·휴가/임금 등 필수 기재사항을 포함해야 함.  
- 적용 가능성: 회사가 선수단에 편입된 장애인 아마추어 선수를 포함해 실질적으로 “근로자”를 상시 10명 이상 사용하는 구조라면, 취업규칙 작성·기재 의무 검토와 연결됨(근로자성 자체를 확정하는 규정은 아님).

- 법률명 및 조항: 근로기준법 제4장 제51조 제1항  
- 조항 내용 요약: 사용자는 취업규칙에 따라 3개월 이내 단위기간을 평균해 주 1주 근로시간을 조정하는 방식 등으로 특정 주·특정 날 근로를 시킬 수 있음(탄력적 근로시간제).  
- 적용 가능성: 훈련·대회 준비가 사실상 근로시간의 배치·조정 문제로 평가될 경우, 회사가 어떤 근로시간제 운영(취업규칙 요건 등)을 했는지가 근로자성 인정 시 추가로 쟁점이 될 수 있음.

- 법률명 및 조항: 근로기준법 제2장 제17조 제1항  
- 조항 내용 요약: 사용자는 근로계약 체결 시 근로자에게 임금, 소정근로시간, 휴일, 연차유급휴가 등 근로조건을 명시해야 함.  
- 적용 가능성: “회사-선수”가 근로계약서를 체결했다는 전제가 있다면, 실제로 근로조건이 근로기준법상 방식으로 명시되었는지(그리고 그 관계가 근로자성에 해당하는지) 판단에 간접적으로 연결됨.

- 법률명 및 조항: 근로기준법 제2

In [19]:
from IPython.display import Markdown, display

display(
    Markdown(result["final_answer"])
)

# 관련 법률

- 법률명 및 조항: 파견근로자 보호 등에 관한 법률(파견법) 제3장 제34조 제1항  
- 조항 내용 요약: 파견 중인 근로자에 대해 파견사업주와 사용사업주를 「근로기준법」상 사용자로 보아, 그에 맞게 「근로기준법」을 적용하되 사용자 판단을 주체별로 구분함.  
- 적용 가능성: “선수단 활동”이 형식상 도급/위탁처럼 보여도 실질이 근로자 파견 또는 사용종속관계로 평가될 여지가 있으면, 「근로기준법」상 사용자 판단과 적용범위 판단에 참고됨(선수=근로자 자동 단정은 아님).

- 법률명 및 조항: 근로기준법 제9장 제93조 제1항  
- 조항 내용 요약: 상시 10명 이상 근로자를 사용하는 사용자는 취업규칙을 작성·신고해야 하며, 근로시간/휴일·휴가/임금 등 필수 기재사항을 포함해야 함.  
- 적용 가능성: 회사가 선수단에 편입된 장애인 아마추어 선수를 포함해 실질적으로 “근로자”를 상시 10명 이상 사용하는 구조라면, 취업규칙 작성·기재 의무 검토와 연결됨(근로자성 자체를 확정하는 규정은 아님).

- 법률명 및 조항: 근로기준법 제4장 제51조 제1항  
- 조항 내용 요약: 사용자는 취업규칙에 따라 3개월 이내 단위기간을 평균해 주 1주 근로시간을 조정하는 방식 등으로 특정 주·특정 날 근로를 시킬 수 있음(탄력적 근로시간제).  
- 적용 가능성: 훈련·대회 준비가 사실상 근로시간의 배치·조정 문제로 평가될 경우, 회사가 어떤 근로시간제 운영(취업규칙 요건 등)을 했는지가 근로자성 인정 시 추가로 쟁점이 될 수 있음.

- 법률명 및 조항: 근로기준법 제2장 제17조 제1항  
- 조항 내용 요약: 사용자는 근로계약 체결 시 근로자에게 임금, 소정근로시간, 휴일, 연차유급휴가 등 근로조건을 명시해야 함.  
- 적용 가능성: “회사-선수”가 근로계약서를 체결했다는 전제가 있다면, 실제로 근로조건이 근로기준법상 방식으로 명시되었는지(그리고 그 관계가 근로자성에 해당하는지) 판단에 간접적으로 연결됨.

- 법률명 및 조항: 근로기준법 제2장 제17조 제2항  
- 조항 내용 요약: 임금 구성항목·계산방법·지급방법 등은 서면으로 교부해야 함(예외·절차 존재).  
- 적용 가능성: 근로자성이 인정될 경우, 서면교부 등 근로조건 명시 절차 준수 여부가 쟁점이 될 수 있음.

- 법률명 및 조항: 노동조합 및 노동관계조정법 제3장 제33조 제1항  
- 조항 내용 요약: 단체협약 기준에 위반하는 취업규칙 또는 근로계약의 부분은 무효이며, 무효로 된 부분은 단체협약 기준을 따름.  
- 적용 가능성: 선수단 운영 규정이 근로계약/취업규칙 또는 그에 준하는 성격으로 평가되고, 단체협약이 존재한다면 단체협약 기준 위반 여부가 추가 쟁점이 될 수 있음(근로자성 확정 규정은 아님).

# 관련 판례

- 판례명: 대법원 2007. 9. 7. 선고 2006도777 판결 [근로기준법위반]  
- 출처: 2006도777 / (판례 파일명: 2006도777) / 분류: 근로자성(근로기준법위반)  
- 판결 요지: 근로자성은 계약 명칭이 아니라 실질적으로 사용자와의 종속적 관계에서 임금을 목적으로 근로를 제공하는지에 따라 판단한다. 시간·장소 지정, 출·퇴근 관리, 전담성, 대체 제한, 보수의 고정적 성격 등은 근로자성 판단 요소가 된다.  
- 사용자 사례와의 관련성: 회사가 장애인 아마추어 운동선수와 “근로계약서”를 체결했더라도 형식이 선수/위촉으로 되어 있을 수 있으나, 회사가 훈련·대회 출전 등 업무 내용을 실질적으로 정하고 시간·장소(또는 선수단 운영 일정)를 관리하며 대체가 제한되고, 대가가 외부 변수와 무관하게 고정·일정 방식으로 지급되는 구조라면 근로자성(따라서 근로기준법 적용) 인정 가능성이 커짐.

# 결론

- 현재 정보 기준 분석 결과:  
  회사가 선수단 운영을 통해 장애인 아마추어 운동선수의 활동을 실질적으로 지휘·감독하고, 선수(근로자)에게 종속적인 방식으로 시간·장소/운영 일정 등을 관리하며, 임금(대가)이 고정적·일정한 방식으로 지급되고, 대체(제3자 대행)가 제한되는 등 종속성과 임금 목적이 인정되면 「근로기준법」상 근로자성 및 사용자 적용 가능성이 높아짐. 특히 “근로계약서” 형식이더라도 명칭이 선수 활동이라면 2006도777 판례 기준의 실질 판단이 핵심임.

- 주요 근거 조항 재정리 (법률명 + 조번호)  
  - 파견근로자 보호 등에 관한 법률(파견법) 제34조 제1항: 사용자를 특정하여 근로기준법을 적용하는 틀(실질 파견/사용종속 평가에 참고)  
  - 근로기준법 제17조 제1항: 근로조건 명시 의무(근로자성이 인정될 경우 연결)  
  - 근로기준법 제17조 제2항: 임금 관련 서면교부(근로자성이 인정될 경우 연결)  
  - 근로기준법 제51조 제1항: 탄력적 근로시간제 운영 요건(근로자성이 인정될 경우 근로시간 배치 쟁점에 참고)  
  - 근로기준법 제93조 제1항: 10명 이상 취업규칙 작성·기재사항(근로자 수 산정 및 운영체계 쟁점에 참고)  
  - 노동조합 및 노동관계조정법 제33조 제1항: 단체협약 위반 취업규칙/근로계약 무효(근로자성 인정 시 근로조건 효력 쟁점에 참고)  
  - 대법원 2006도777(2007.9.7): 계약 명칭이 아니라 종속성과 임금 목적 등 실질로 근로자성 판단